# Pricing Asian Style Options

This notebook follows QMCPy's `pricing_options.ipynb` as closely as practical while keeping the Julia example lightweight enough for routine execution.

Original QMCPy demo: [`QMCPy/demos/pricing_options.ipynb`](../../QMCPy/demos/pricing_options.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMC.jl/blob/develop/demos/pricing_options.ipynb)

Parity note: this Julia notebook keeps the same section progression and option families from QMCPy, namely the European, arithmetic-mean, geometric-mean, barrier, and lookback examples, with the same seed choices where applicable. The main deliberate divergence is the smaller sample budget (`100_000` here versus `10^6` in QMCPy), together with Julia-native helper code and exact-value hooks that keep the checked-in demo lightweight.

As in the Python demo, the focus is on Monte Carlo pricing for path-dependent options with Asian-style payoffs and European exercise.

- The payoff depends on the whole asset price path, not only on the terminal asset price.
- The option is only exercised at expiry, unlike American options, which can be exercised at any time before expiry.


In [1]:
using QMC
using Printf
using Statistics

function asian_trapezoidal_integrand(dd; call_put=:call, geometric=false, d=12, S0=120.0, r=0.02, σ=0.5, K=130.0, T=1 / 4)
    tm = GeometricBrownianMotion(dd; t_final=T, initial_value=S0, drift=r, diffusion=σ^2)
    return CustomFun(tm, x -> begin
        if geometric
            avg = exp.((log(S0) / 2 .+ vec(sum(log.(x[:, 1:(end - 1)]), dims=2)) .+ log.(x[:, end]) / 2) ./ d)
        else
            avg = (S0 / 2 .+ vec(sum(x[:, 1:(end - 1)], dims=2)) .+ x[:, end] / 2) ./ d
        end
        payoff = call_put == :call ? max.(avg .- K, 0.0) : max.(K .- avg, 0.0)
        vec(exp(-r * T) .* payoff)
    end)
end


asian_trapezoidal_integrand (generic function with 1 method)

In [2]:
S0 = 120.0
r = 0.02
σ = 0.5
K = 130.0
T = 1 / 4
d = 12
sample_size = 100_000


100000

## European Options


In [3]:
tm = BrownianMotion(IIDStdUniform(d; seed=7); t_final=T)
eu_call = FinancialOption(tm; option_type=:european, call_put=:call, volatility=σ, start_price=S0, strike_price=K, interest_rate=r)
y = sample_and_evaluate(eu_call, sample_size)
@printf("The exact price of this European Call Option is %.4f
", get_exact_value(eu_call))
@printf("After generating %d iid points, the price estimate is %.4f
", sample_size, mean(y))


The exact price of this European Call Option is 8.2779
After generating 100000 iid points, the price estimate is 8.3678


## Arithmetic Mean Options

The payoff of the arithmetic-mean Asian option depends on the average stock price over the monitoring dates rather than only on the terminal value. To stay close to the QMCPy demo, the Julia notebook uses the same trapezoidal averaging convention for the path average.

$$\begin{array}{rcc}
 & \textbf{call} & \textbf{put} \\ \hline
\textbf{payoff} &
\displaystyle \max\!\left(\frac 1d \sum_{j=1}^d S(jT/d) - K, 0\right)e^{-rT} &
\displaystyle \max\!\left(K - \frac 1d \sum_{j=1}^d S(jT/d), 0\right)e^{-rT}
\end{array}$$


In [4]:
arith_call = asian_trapezoidal_integrand(IIDStdUniform(d; seed=7); call_put=:call, geometric=false, d=d, S0=S0, r=r, σ=σ, K=K, T=T)
y = sample_and_evaluate(arith_call, sample_size)
@printf("After generating %d iid points, the price of this Arithmetic Mean Call Option is %.4f
", sample_size, mean(y))


After generating 100000 iid points, the price of this Arithmetic Mean Call Option is 3.4366


The price of the Asian arithmetic mean call option is smaller than the price of the European call option.

We may also price the Asian arithmetic mean put option as follows:


In [5]:
arith_put = asian_trapezoidal_integrand(IIDStdUniform(d; seed=7); call_put=:put, geometric=false, d=d, S0=S0, r=r, σ=σ, K=K, T=T)
y = sample_and_evaluate(arith_put, sample_size)
@printf("After generating %d iid points, the price of this Arithmetic Mean Put Option is %.4f
", sample_size, mean(y))


After generating 100000 iid points, the price of this Arithmetic Mean Put Option is 13.0015


Note that the put price is greater here because the strike is above the initial price.

In the limit of continuous monitoring, the arithmetic average is approximated by taking more time steps:

$$\begin{array}{rcc}
& \textbf{call} & \textbf{put} \\ \hline
\textbf{payoff} &
\displaystyle \max\!\left(\frac 1T \int_0^T S(t) \, \mathrm{d}t - K, 0\right)e^{-rT} &
\displaystyle \max\!\left(K - \frac 1T \int_0^T S(t) \, \mathrm{d}t, 0\right)e^{-rT}
\end{array}$$


In [6]:
arith_call_fine = asian_trapezoidal_integrand(IIDStdUniform(62; seed=7); call_put=:call, geometric=false, d=62, S0=S0, r=r, σ=σ, K=K, T=T)
y = sample_and_evaluate(arith_call_fine, sample_size)
@printf("After generating %d iid points, the price of this Arithmetic Mean Call Option is %.4f
", sample_size, mean(y))


After generating 100000 iid points, the price of this Arithmetic Mean Call Option is 3.4409


The price is a bit lower, and the runtime is longer because more monitoring dates require more random variables.


## Geometric Mean Options

One can also base the payoff on a geometric mean rather than an arithmetic mean. QMC.jl provides this case directly through `FinancialOption`, together with an exact value for the discrete geometric-average variant used below. This keeps the reported prices close to the corresponding closed-form values shown in the QMCPy demo.

As in QMCPy, the geometric mean is no larger than the arithmetic mean, so geometric-average call options are cheaper and geometric-average put options are more expensive than their arithmetic counterparts.


In [7]:
tm = BrownianMotion(IIDStdUniform(d; seed=7); t_final=T)
geo_put = FinancialOption(tm; option_type=:asian, call_put=:put, mean_type=:geometric, volatility=σ, start_price=S0, strike_price=K, interest_rate=r)
y = sample_and_evaluate(geo_put, sample_size)
@printf("The exact price of this Geometric Asian Put Option is %.4f\n", get_exact_value(geo_put))
@printf("After generating %d iid points, the price of this Geometric Mean Put Option is %.4f\n", sample_size, mean(y))


The exact price of this Geometric Asian Put Option is 13.7841
After generating 100000 iid points, the price of this Geometric Mean Put Option is 13.7434


In [8]:
tm = BrownianMotion(IIDStdUniform(d; seed=7); t_final=T)
geo_call = FinancialOption(tm; option_type=:asian, call_put=:call, mean_type=:geometric, volatility=σ, start_price=S0, strike_price=K, interest_rate=r)
y = sample_and_evaluate(geo_call, sample_size)
@printf("The exact price of this Geometric Asian Call Option is %.4f\n", get_exact_value(geo_call))
@printf("After generating %d iid points, the price of this Geometric Mean Call Option is %.4f\n", sample_size, mean(y))


The exact price of this Geometric Asian Call Option is 3.5401
After generating 100000 iid points, the price of this Geometric Mean Call Option is 3.5901


## Barrier Option

In a barrier option, the payoff only occurs if the asset price crosses a barrier. Here we price an up-and-in call, matching the qualitative setup of the QMCPy demo.

$$\begin{array}{rcc}
 & \textbf{up}\ (S(0) < b) & \textbf{down}\ (S(0) > b) \\ \hline
 \textbf{in} & \text{active if } S(t) \ge b & \text{active if } S(t) \le b \\
 \textbf{out} & \text{inactive if } S(t) \ge b & \text{inactive if } S(t) \le b
\end{array}$$


In [9]:
tm = BrownianMotion(IIDStdUniform(d; seed=7); t_final=T)
barrier_call = FinancialOption(tm; option_type=:barrier, call_put=:call, volatility=σ, start_price=S0, strike_price=K, interest_rate=r, barrier_price=150.0, barrier_in_out=:in)
y = sample_and_evaluate(barrier_call, sample_size)
@printf("After generating %d iid points, the price of this Barrier UpIn Call Option is %.4f
", sample_size, mean(y))


After generating 100000 iid points, the price of this Barrier UpIn Call Option is 7.4949


Note that this price is less than the European call option because the asset price must cross the barrier for the option to become active.


## Lookback Options

Lookback options use the running path extrema rather than a fixed strike level in the payoff. As in the QMCPy demo, the payoff depends on the whole asset-price path.

$$\begin{array}{rcc}
& \textbf{call} & \textbf{put} \\ \hline
\textbf{payoff} &
\displaystyle \max\!\left(S(T) - \min_{0 \le t \le T} S(t), 0\right)e^{-rT} &
\displaystyle \max\!\left(\max_{0 \le t \le T} S(t) - S(T), 0\right)e^{-rT}
\end{array}$$


In [10]:
tm = BrownianMotion(IIDStdUniform(d; seed=7); t_final=T)
look_call = FinancialOption(tm; option_type=:lookback, call_put=:call, volatility=σ, start_price=S0, strike_price=K, interest_rate=r)
y = sample_and_evaluate(look_call, sample_size)
@printf("After generating %d iid points, the price of this Lookback Call Option is %.4f
", sample_size, mean(y))


After generating 100000 iid points, the price of this Lookback Call Option is 13.7344
